# ARBench on Google Colab (L4 GPU)

Benchmarks **Granite 4.1, Gemma 4 and Qwen 3** (via Ollama) on your annual report:
YourBench-style benchmark generation → LightEval-style evaluation → per-model + comparison reports.

**Before you start:** `Runtime → Change runtime type → L4 GPU`.

Steps: 1) get the project · 2) install Ollama · 3) add the annual report · 4) run (one command) · 5) download results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Get the project

**Option A (recommended):** upload the whole project folder to Google Drive (e.g. `MyDrive/ARBench`) and set `PROJECT_DIR`.
**Option B:** zip the project folder on your laptop (without `.venv` and `outputs`) and upload the zip when prompted.

In [ ]:
import os, zipfile, glob
USE_DRIVE = True                                   # False -> upload a zip instead
PROJECT_DIR = "/content/drive/MyDrive/ARBench"     # folder that contains run_benchmark.py

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    up = files.upload()                            # choose ARBench.zip
    zname = next(iter(up))
    zipfile.ZipFile(zname).extractall("/content/ARBench_src")
    hits = glob.glob("/content/ARBench_src/**/run_benchmark.py", recursive=True)
    PROJECT_DIR = os.path.dirname(hits[0])

assert os.path.exists(os.path.join(PROJECT_DIR, "run_benchmark.py")), f"run_benchmark.py not found in {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Project:", os.getcwd())

## 2 · Install Ollama + Python dependencies and start the server (GPU)

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install -r requirements.txt

In [ ]:
import subprocess, time, requests, os
env = dict(os.environ,
           OLLAMA_NUM_PARALLEL="4",        # parallel requests per model (matches --profile colab)
           OLLAMA_MAX_LOADED_MODELS="3",   # all three SLMs fit in 24 GB VRAM
           OLLAMA_KEEP_ALIVE="30m",
           OLLAMA_FLASH_ATTENTION="1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama", requests.get("http://127.0.0.1:11434/api/version", timeout=2).json()); break
    except Exception:
        time.sleep(1)

## 3 · Choose the models and add the annual report

The defaults are the same small models as the laptop run. On the L4 you can also try larger sizes of the same families.

In [ ]:
MODELS = ["granite4.1:3b", "gemma4:e2b-it-qat", "qwen3:4b-instruct"]
# Larger variants that fit an L4 comfortably:
# MODELS = ["granite4.1:8b", "gemma4:e4b-it-qat", "qwen3:8b"]
for m in MODELS:
    !ollama pull {m}

In [ ]:
import os, shutil
REPORT_DIR = "data/annual_report"
os.makedirs(REPORT_DIR, exist_ok=True)
existing = [f for f in os.listdir(REPORT_DIR) if f.lower().endswith((".pdf", ".docx", ".html", ".htm", ".md", ".txt")) and f.lower() != "readme.md"]
if not existing:
    from google.colab import files
    print("Upload the annual report (PDF/DOCX/HTML/MD/TXT):")
    for name, data in files.upload().items():
        open(os.path.join(REPORT_DIR, name), "wb").write(data)
print("Reports:", [f for f in os.listdir(REPORT_DIR) if f.lower() != "readme.md"])

## 4 · Run the whole benchmark (one command)

`--profile colab` = full coverage (≈60 single-hop chunks, multi-hop, MCQ) with 4 parallel requests.
Use `--profile quick` for a fast first look. Everything is cached, so re-running the cell resumes.

In [ ]:
!python run_benchmark.py --profile colab --run-name colab_run --models {" ".join(MODELS)}

## 5 · Results

In [ ]:
from IPython.display import Markdown, display, HTML
run = "outputs/colab_run"
display(Markdown(open(f"{run}/reports/comparison_report.md", encoding="utf-8").read().replace("](charts/", f"]({run}/reports/charts/")))

In [ ]:
from IPython.display import Image
for c in ["leaderboard", "tasks", "quality_vs_speed", "hallucination", "question_types", "difficulty"]:
    p = f"outputs/colab_run/reports/charts/{c}.png"
    if os.path.exists(p): display(Image(p))

In [ ]:
import shutil
shutil.make_archive("/content/arbench_results", "zip", "outputs/colab_run")
from google.colab import files
files.download("/content/arbench_results.zip")   # also saved inside PROJECT_DIR/outputs if you used Drive